# Week 5 Project: Tree-Based Models & Ensembles
**Statistical Machine Learning · Fusemachines AI Fellowship · Facilitator: Susan Ghimire**

**Deliverables:** `W5_Assignment.ipynb` (this notebook), `telco_churn_pipeline_v1.joblib` (saved pipeline)

## 1 YOUR ROLE
You are a Data Scientist at a telecom company. The business is losing customers to churn and needs a production-ready predictive system — not a demo. The analytics team previously tried basic linear models but they failed to capture the complex, non-linear relationships in customer behaviour. You have one week to upgrade the system using tree-based ensembles.

**Responsibilities (summary):** Build robust ensembles, prevent data leakage (use pipelines), tune models systematically, generate SHAP explanations, and save the full fitted pipeline for production.

## 2 DATASET
Dataset: Telco Customer Churn (Kaggle). 7,043 rows · 21 columns. Primary target: `Churn` (Yes/No).

**Data issues to handle:**
- `TotalCharges` may be read as object; coerce with `pd.to_numeric(..., errors='coerce')`.
- Class imbalance (~27% churn). Use SMOTE inside an `imblearn` pipeline during CV.
- Mixed dtypes: use `ColumnTransformer` to apply different preprocessing to numeric vs categorical features.

## 3 HOW TO USE THIS NOTEBOOK
1. Run cells top-to-bottom in order.
2. Replace any `YOUR CODE HERE` placeholders with working code.
3. Ensure all `SELF-CHECK` asserts pass before submission.

## 4 SUBMISSION REQUIREMENTS
Submit: `W5_Assignment.ipynb` (executed, all outputs visible) and `telco_churn_pipeline_v1.joblib` (serialised pipeline).

## 5 COMMON MISTAKES TO AVOID (quick reference)
- Use `imblearn.pipeline.Pipeline` when SMOTE is a step.
- Do not fit scalers on the full dataset before splitting.
- Save the full pipeline (including preprocessing and SMOTE) with `joblib`.

In [ ]:
# Install (Colab) and imports. Uncomment the pip lines if running in a fresh environment.
# !pip install xgboost imbalanced-learn shap joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import shap
import joblib

In [ ]:
# Load dataset (update path if needed)
# Replace 'path/to/Telco-Customer-Churn.csv' with your CSV location
df = pd.read_csv('Telco-Customer-Churn.csv')
df.shape, df.columns.tolist()

In [ ]:
# Data cleaning: TotalCharges dtype bug and basic cleanup
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
# Inspect rows with NaN TotalCharges (new customers)
df[df['TotalCharges'].isna()].head()
# Option 1: impute with 0 or MonthlyCharges * tenure (choose one and document)
# Example: impute with 0 for simplicity here (consider alternative in your writeup)
df['TotalCharges'] = df['TotalCharges'].fillna(0)
# Convert target to binary
df['Churn_flag'] = (df['Churn'] == 'Yes').astype(int)
# Drop customerID if present
if 'customerID' in df.columns:
    df = df.drop(columns=['customerID'])
df.shape

In [ ]:
# Split features/target and create train/test split BEFORE any preprocessing
X = df.drop(columns=['Churn', 'Churn_flag'])
y = df['Churn_flag']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train.shape, X_test.shape, y_train.mean(), y_test.mean()

In [ ]:
# Preprocessing: identify numeric and categorical features
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = [c for c in X.columns if c not in numeric_features]
numeric_features, categorical_features[:10]

# Numeric pipeline
from sklearn.pipeline import make_pipeline
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())

In [ ]:
# Categorical pipeline
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))

In [ ]:
# ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('cat', categorical_transformer, categorical_features)

In [ ]:
# Example pipeline with SMOTE (use ImbPipeline) and XGBoost classifier.
from imblearn.over_sampling import SMOTE
estimator = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
pipeline = ImbPipeline(steps=[
    ('smote', SMOTE(random_state=42)),
])
: 
,
: { 
: 
 },
: [
,
3
6
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
gsearch = GridSearchCV(pipeline, param_grid, scoring='roc_auc', cv=cv, n_jobs=-1, verbose=2)
# Fit on training data only
# gsearch.fit(X_train, y_train)
# Best params: gsearch.best_params_

In [ ]:
# Evaluation on test set (after training). Replace `model` with fitted pipeline (gsearch.best_estimator_ or pipeline.fit)
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:,1]
    return {
        'precision': precision_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred)

: 
,
: { 
: 
 },
: [
,
,